In [0]:
%run ./_config

In [0]:
# simple pipeline health report
health = spark.sql(f"""
    SELECT
        (SELECT COUNT(*) FROM {SILVER_SCHEMA}.fct_sales)         AS fact_rows_raw,
        (SELECT COUNT(*) FROM {SILVER_SCHEMA}.fct_sales_deduped) AS fact_rows_clean,
        (SELECT COUNT(*) FROM {SILVER_SCHEMA}.fct_sales)
            - (SELECT COUNT(*) FROM {SILVER_SCHEMA}.fct_sales_deduped) AS duplicate_count,
        (SELECT COUNT(DISTINCT store_id)  FROM {SILVER_SCHEMA}.dim_stores  WHERE __END_AT IS NULL) AS active_stores,
        (SELECT COUNT(DISTINCT item_id)   FROM {SILVER_SCHEMA}.dim_products WHERE __END_AT IS NULL) AS active_products,
        (SELECT COUNT(DISTINCT vendor_id) FROM {SILVER_SCHEMA}.dim_vendors  WHERE __END_AT IS NULL) AS active_vendors,
        (SELECT MIN(sale_date) FROM {SILVER_SCHEMA}.fct_sales_deduped) AS earliest_sale,
        (SELECT MAX(sale_date) FROM {SILVER_SCHEMA}.fct_sales_deduped) AS latest_sale
""")
health.display()

In [0]:
# read upstream task values (works only within Workflow)
try:
    rows_ingested = dbutils.jobs.taskValues.get(
        taskKey="01_ingest_api",
        key="rows_ingested",
        default=0
    )
    load_date = dbutils.jobs.taskValues.get(
        taskKey="01_ingest_api",
        key="load_date",
        default="unknown"
    )
    print(f"Ingestion reported: {rows_ingested:,} rows, load date: {load_date}")
except Exception:
    print("Task values unavailable")

In [0]:
# Cell 4 — Gold layer validation
gold_tables = [
    "gold_monthly_sales", "gold_dow_patterns", "gold_county_performance",
    "gold_city_rankings", "gold_vendor_scorecard",
    "gold_category_market_share", "gold_store_rankings"
]

for tbl in gold_tables:
    count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {GOLD_SCHEMA}.{tbl}").collect()[0]["cnt"]
    status = "OK" if count > 0 else "Empty, check pipeline"
    print(f"  {tbl}: {count:,} rows — {status}")